# Action Recognition

### Why Single Frames Aren't Enough

Everything so far looked at one frame at a time:
- YOLO → what's in this frame?
- tracking → same object across frames?
- optical flow → where did pixels move?

Action recognition is different:
- one frozen frame of a person with leg raised could be walking, kicking, dancing, climbing
- you need multiple frames over time to know which action it is

The input is no longer a single image — it's a **sequence of frames**.

### Why A Standard CNN Isn't Enough

CNNs understand spatial features:
- what's where in a single image
- edges, textures, shapes

CNNs have no concept of time:
- stacking 16 frames into one input loses order
- model can't tell frame 5 came before frame 6
- no understanding of how things changed

We need a model that understands both **space and time**.

### 3D CNN — Adding Time

A regular CNN uses 2D filters that slide over height and width:

```
2D filter:         3D filter:
┌───┐              ┌───┬───┬───┐
│   │              │ t │ t │ t │  ← slides through time too
└───┘              └───┴───┴───┘
slides over        slides over
height × width     height × width × time
```

3D CNN naturally understands what changed between frames:
- no need to manually encode timestamps
- order is preserved by the time dimension
- spatial and temporal features learned simultaneously

### Input Shape

A single image shape:

```
(3, 224, 224)
 ↑    ↑    ↑
 │    │    └── width  (pixels)
 │    └─────── height (pixels)
 └──────────── channels (3 = RGB)
```

A video clip shape for 3D CNN:

```
(3, 16, 224, 224)
 ↑   ↑    ↑    ↑
 │   │    │    └── width
 │   │    └─────── height
 │   └──────────── frames (16 frames ≈ 0.5 seconds at 30fps)
 └──────────────── channels (RGB)
```

With batch dimension added:

```
(1, 3, 16, 224, 224)
 ↑
 batch size
```

### Sliding Window

The model processes 16 frames at a time using a sliding window:

```
Frames:   1  2  3  4  5  6  7  8 ... 16  17  18  19  20

Window 1: [1  2  3  4  5  6  7  8 ... 16]
Window 2:        [5  6  7  8  9  10 ... 20]
Window 3:                 [9  10  11 ... 24]
```

Window slides forward every `STEP` frames — same concept as `frame % DETECT_EVERY` in YOLO:
- process every 5th frame → `frame % 5 == 0`
- slide window every 5 frames → same thing, applied to a clip

Label updates smoothly as window slides:
```
frames 1–16   → "walking"
frames 5–20   → "walking"
frames 9–24   → "running"  ← action changed
```

### Why We Crop The Person First

Feeding the full frame to R3D causes poor predictions:
- background, furniture, walls all influence the prediction
- Kinetics-400 clips are centered on the person performing the action
- full webcam frame looks nothing like training data

Fix:
- detect person with YOLO
- crop just the person bounding box
- feed crop to R3D

```
Full frame → YOLO → person box → crop → R3D → action label
```

Crops vary slightly in size each frame as the person moves. <br>
This is fine — preprocessing resizes every crop to `(112, 112)` anyway.

### Buffer Management

R3D needs 16 consecutive crops of the same person.

Problem — what if YOLO misses the person for a few frames?

Two scenarios:

```
Scenario A — brief miss (YOLO glitch):
detected, detected, MISSED, detected, detected
→ keep buffer, wait it out

Scenario B — person left frame and came back:
guitar, guitar, GONE x 30 frames, soccer, soccer
→ buffer must reset — mixed clip = nonsense prediction
```

Solution — tolerance counter:
- person missing for ≤ `MAX_MISS` frames → keep buffer
- person missing for > `MAX_MISS` frames → clear buffer, fresh start

Same idea as DeepSORT remembering an ID for a few frames before declaring it lost.

### Kinetics-400 — What R3D Already Knows

R3D comes pretrained on **Kinetics-400**:
- 400 action classes
- 240,000 video clips
- actions like: walking, running, jumping, clapping, cooking, playing guitar

Same idea as COCO for YOLO — pretrained weights, no training needed.

### Setup

In [17]:
import cv2
import torch
import numpy as np
import random
import torch.nn.functional as F
from ultralytics import YOLO
from torchvision.models.video import r3d_18, R3D_18_Weights
from torchvision.transforms import Compose, Resize, CenterCrop, Normalize

#### `r3d_18`
- `R3D` = Residual 3D network
- `18` = 18 layers deep
- same ResNet architecture from Face Recognition, extended to 3D

#### `R3D_18_Weights`
- pretrained weight configuration for R3D-18
- downloads weights automatically on first run

#### `CenterCrop`
- crops the center region of the image
- removes edges which often contain less useful information

#### `Normalize`
- scales pixel values to a standard range
- required to match the distribution the model was trained on

### Constants

In [ ]:
CLIP_LENGTH    = 16     # frames per clip fed to R3D
STEP           = 8      # slide window every N frames
TOP_K          = 3      # show top K predicted actions
CONF_THRESHOLD = 0.5    # YOLO confidence threshold
DETECT_EVERY   = 5      # run YOLO every N frames
MAX_MISS       = 10     # frames without detection before buffer resets
device         = "cuda" if torch.cuda.is_available() else "cpu"

# class coloring
random.seed(42)
COLORS = {
    class_id: tuple(random.randint(50, 255) for _ in range(3))
    for class_id in range(80)
}

| Constant | Meaning |
| -------- | ------- |
| `CLIP_LENGTH` | number of frames per inference window |
| `STEP` | how many frames to slide before next R3D inference |
| `TOP_K` | number of top predictions to display |
| `DETECT_EVERY` | run YOLO every N frames |
| `MAX_MISS` | frames without detection before buffer resets |

### Load Models

In [19]:
# YOLO
yolo = YOLO('yolov8n.pt')

# R3D
weights = R3D_18_Weights.DEFAULT
r3d     = r3d_18(weights=weights).eval().to(device)
labels  = weights.meta["categories"]

#### `weights.meta["categories"]`
- list of 400 Kinetics action class names
- index matches model output — same as `model.names` in YOLO

### Preprocessing

In [20]:
preprocess = Compose([
    Resize((128, 171)),
    CenterCrop((112, 112)),
    Normalize(mean=[0.43216, 0.394666, 0.37645],
              std=[0.22803, 0.22145, 0.216989])
])

Why these specific values?
- `mean` and `std` were computed across the entire Kinetics-400 dataset
- normalizing with the same values the model trained on keeps pixel distributions consistent
- using different values would degrade predictions

Pipeline per crop:
```
person crop (variable size)
↓
Resize → (128×171)      ← shrink, preserve aspect ratio
↓
CenterCrop → (112×112)  ← square crop from center
↓
Normalize               ← match Kinetics-400 distribution
```

### Helper Functions

In [21]:
def get_person_crop(frame, yolo_results):
    """Returns the largest person crop from YOLO detections."""

    best_box  = None
    best_area = 0

    for box in yolo_results[0].boxes:
        class_id = int(box.cls.item())
        if yolo.names[class_id] != "person":
            continue
        if box.conf.item() < CONF_THRESHOLD:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        area = (x2 - x1) * (y2 - y1)

        if area > best_area:
            best_area = area
            best_box  = (x1, y1, x2, y2)

    if best_box is None:
        return None, None

    x1, y1, x2, y2 = best_box
    crop = frame[y1:y2, x1:x2]

    if crop.size == 0:
        return None, None

    return crop, best_box

#### Why largest person?
- multiple people may be detected
- largest box = closest to camera = most detail
- most likely the subject of interest

In [22]:
def prepare_clip(crops):
    """Converts list of BGR crops into R3D input tensor."""

    processed = []

    for crop in crops:
        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        tensor   = torch.tensor(crop_rgb).permute(2, 0, 1).float() / 255.0
        tensor   = preprocess(tensor)
        processed.append(tensor)

    # Stack frames: list of (3,112,112) → (16,3,112,112) → (3,16,112,112)
    clip = torch.stack(processed, dim=0).permute(1, 0, 2, 3)

    # Add batch dimension: (3,16,112,112) → (1,3,16,112,112)
    return clip.unsqueeze(0).to(device)

#### `permute(2, 0, 1)`
- OpenCV frames are shape `(H, W, C)` — height, width, channels last
- PyTorch expects `(C, H, W)` — channels first
- moves channel axis from position 2 to position 0

#### `/ 255.0`
- converts pixel values from `0–255` to `0.0–1.0`
- required before normalization

#### `torch.stack(processed, dim=0).permute(1, 0, 2, 3)`
- `stack` combines 16 tensors of `(3,112,112)` into `(16,3,112,112)`
- `permute(1,0,2,3)` reorders to `(3,16,112,112)` — channels first, then time

#### `unsqueeze(0)`
- adds batch dimension
- `(3,16,112,112)` → `(1,3,16,112,112)`

In [ ]:
def predict_action(crops):
    """Runs R3D on a clip of crops and returns top K predictions."""

    clip = prepare_clip(crops)

    with torch.no_grad():
        output = r3d(clip)

    probs = F.softmax(output[0], dim=0)
    top_probs, top_indices = torch.topk(probs, TOP_K)

    return [(labels[idx.item()], prob.item()) for prob, idx in zip(top_probs, top_indices)]

#### `F.softmax(output[0], dim=0)`
- `softmax` = converts raw model scores to probabilities summing to 1.0
- `dim=0` applies across the 400 class scores

#### `torch.topk(probs, TOP_K)`
- returns K highest probabilities and their indices
- `top_probs` = confidence values
- `top_indices` = class indices → mapped to names via `labels[idx]`

### Webcam Loop

In [ ]:
cap = cv2.VideoCapture(0)

crop_buffer       = []
stored_detections = []
current_preds     = []
current_box       = None
frame_count       = 0
miss_count        = 0

while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Detection
    if frame_count % DETECT_EVERY == 0:
        results = yolo(frame, conf=CONF_THRESHOLD, verbose=False)
        crop, box = get_person_crop(frame, results)

        if crop is not None:
            miss_count  = 0
            current_box = box
            crop_buffer.append(crop)

            if len(crop_buffer) > CLIP_LENGTH:
                crop_buffer.pop(0)

        else:
            miss_count += 1
            if miss_count > MAX_MISS:
                crop_buffer = []
                current_preds = []
                current_box = None
                miss_count = 0

    # Action recognition
    if len(crop_buffer) == CLIP_LENGTH and frame_count % STEP == 0:
        current_preds = predict_action(crop_buffer)

    # Draw person box
    if current_box is not None:
        x1, y1, x2, y2 = current_box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # Draw predictions
    for i, (label, prob) in enumerate(current_preds):
        text = f"{label}: {prob:.2f}"
        cv2.putText(frame, text, (20, 40 + i * 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # Buffer progress
    cv2.putText(frame, f"Buffer: {len(crop_buffer)}/{CLIP_LENGTH}", (20, 460),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.imshow("Action Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

#### `miss_count`
- counts consecutive frames where no person was detected
- resets to 0 whenever person is found
- triggers buffer reset when exceeds `MAX_MISS`

#### Buffer reset
- clears `crop_buffer` — no mixed action clips
- clears `current_preds` — no stale predictions on screen
- clears `current_box` — no stale bounding box drawn

#### `Buffer: {len(crop_buffer)}/{CLIP_LENGTH}`
- shows how many crops are collected so far
- useful to see when first prediction will appear
- disappears once buffer fills and predictions start

### Summary

| Concept | What It Means |
| ------- | ------------- |
| Action recognition | classifying actions across a sequence of frames |
| 3D CNN | CNN with time as a third dimension |
| `R3D` | Residual 3D — ResNet extended to video |
| Person crop | YOLO box cropped and fed to R3D instead of full frame |
| Largest person | closest to camera — most detail, most likely subject |
| `crop_buffer` | rolling list of last N person crops |
| `miss_count` | tolerance counter before buffer resets |
| `MAX_MISS` | frames without detection before fresh start |
| `permute` | reorders tensor dimensions to match expected shape |
| `softmax` | converts raw scores to probabilities summing to 1.0 |
| `topk` | returns K highest values and their indices |
| Kinetics-400 | dataset R3D was pretrained on — 400 action classes |